In [2]:
import glob
import pandas as pd

csv_files = glob.glob('*.csv')

for filename in csv_files:
    print(f"📄 文件名: {filename}")
    try:
        # 只读取前 2 行 (nrows=1 读取数据行，header默认是第一行)
        # 注意：pd.read_csv 默认把第一行当做表头，nrows=1 会读入表头 + 1行数据
        df = pd.read_csv(filename, nrows=1)
        
        # 打印表头（列名）
        print(f"   第一行 (列名): {list(df.columns)}")
        
        # 打印第一行数据 (如果文件不为空)
        if not df.empty:
            print(f"   第二行 (数值): {df.iloc[0].tolist()}")
            
    except Exception as e:
        print(f"   读取出错: {e}")
    print("-" * 40)

📄 文件名: AD.csv
   第一行 (列名): ['path', 'filename', 'visit', 'age', 'gender', 'apoe', 'NC', 'MCI', 'DE', 'COG', 'AD', 'PD', 'FTD', 'VD', 'DLB', 'PDD', 'ADD', 'OTHER', 'mmse', 'cdr', 'lm_imm', 'lm_del', 'Tesla']
   第二行 (数值): ['/home/wcj/data/AIBL/', 'AIBL_683_MR_MPRAGE_ADNI_confirmed__br_raw_20181004110622741_74_S82565_I169879.nii', 'bl', 81, 'male', 0, 0, 0, 1, 2, 1, nan, nan, nan, nan, nan, 1.0, nan, 19, 1.0, 4, 5, 1.5]
----------------------------------------
📄 文件名: kg.csv
   第一行 (列名): ['relation', 'display_relation', 'x_index', 'x_id', 'x_type', 'x_name', 'x_source', 'y_index', 'y_id', 'y_type', 'y_name', 'y_source']
   第二行 (数值): ['protein_protein', 'ppi', 0, 9796, 'gene/protein', 'PHYHIP', 'NCBI', 8889, 56992, 'gene/protein', 'KIF15', 'NCBI']
----------------------------------------
📄 文件名: MCI.csv
   第一行 (列名): ['path', 'filename', 'visit', 'age', 'gender', 'apoe', 'NC', 'MCI', 'DE', 'COG', 'AD', 'PD', 'FTD', 'VD', 'DLB', 'PDD', 'ADD', 'OTHER', 'mmse', 'cdr', 'lm_imm', 'lm_del', 'Tesla'

In [2]:
import pandas as pd

# 1. 读取两个三元组文件
df_adni = pd.read_csv("adni_knowledge_triplets.csv")
df_prime = pd.read_csv("primekg_ad_only.csv")

# 2. 提取 ADNI 侧试图连接的 "PrimeKG" 开头的尾实体
# (代码中病人连接到 -> PrimeKG:xxxx)
adni_targets = set(df_adni[df_adni['tail'].str.startswith('PrimeKG:')]['tail'].unique())

# 3. 提取 PrimeKG 侧实际存在的实体 (加上前缀以匹配你的格式)
# 注意：你的 primekg_ad_only.csv 里的 x_name 是原始名字 (如 "DHX9")
# 而你的代码里给 ADNI 加上了 "PrimeKG:" 前缀
# 所以我们要看 PrimeKG 文件里的名字是否包含 ADNI 想连的那些名字
prime_nodes = set("PrimeKG:" + df_prime['x_name'].astype(str)) | \
              set("PrimeKG:" + df_prime['y_name'].astype(str))

# 4. 求交集：看有多少能连上
overlap = adni_targets.intersection(prime_nodes)
missing = adni_targets - prime_nodes

print(f"📊 连通性分析:")
print(f"   - ADNI 试图连接的 PrimeKG 节点数: {len(adni_targets)}")
print(f"   - 实际在 PrimeKG 子图中存在的节点数: {len(overlap)}")
print(f"   - ❌ 断链节点 (ADNI指涉了但PrimeKG里没有): {len(missing)}")

if len(missing) > 0:
    print(f"\n⚠️ 警告：以下关键节点可能名字不匹配，导致图谱断裂！")
    print(list(missing)[:10]) # 打印前10个看看
else:
    print("\n✅ 完美！所有桥接节点都已对齐。")

FileNotFoundError: [Errno 2] No such file or directory: 'adni_knowledge_triplets.csv'